In [0]:
# src/notebooks/09_unity_catalog_security.py
# Branch: feature/gold-layer

import pyspark.sql.functions as F

CATALOG = "vstone_catalog"
SECURITY_SCHEMA = f"{CATALOG}.security"
GOLD_SCHEMA = f"{CATALOG}.gold"

# 0. Context Setup
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SECURITY_SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")

# ======================================================================================
# 1. ROW LEVEL SECURITY (RLS) - Fixed for Admin Visibility
# ======================================================================================
print(f" Creating Row Filter Function for {CATALOG}...")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.listing_region_filter(location_key STRING)
RETURNS BOOLEAN
RETURN
  IS_ACCOUNT_GROUP_MEMBER('admins') OR  -- Fix: Allowing your current group
  IS_ACCOUNT_GROUP_MEMBER('admin_group') OR 
  (IS_ACCOUNT_GROUP_MEMBER('moscow_team') AND location_key IN ('Moscow', 'Podolsk', 'Khimki', 'Mytishchi')) OR
  (IS_ACCOUNT_GROUP_MEMBER('west_team') AND location_key IN ('Saint-Petersburg', 'Kazan', 'Nizhny-Novgorod', 'Samara', 'Krasnodar')) OR
  (IS_ACCOUNT_GROUP_MEMBER('east_team') AND location_key IN ('Novosibirsk', 'Krasnoyarsk', 'Irkutsk', 'Vladivostok', 'Khabarovsk')) OR
  IS_ACCOUNT_GROUP_MEMBER('analyst_group')
""")

# Applying RLS to Gold tables
for table in ["fact_listings_liquid", "agg_stale_inventory"]:
    spark.sql(f"ALTER TABLE {GOLD_SCHEMA}.{table} SET ROW FILTER {SECURITY_SCHEMA}.listing_region_filter ON (location_key)")
    print(f"🛡️ RLS applied to {GOLD_SCHEMA}.{table}")

# ======================================================================================
# 2. COLUMN LEVEL SECURITY (CLS) - Dynamic Masking
# ======================================================================================
print("\n🚀 Implementing Column-Level Masking...")

# MASK: Listing ID
spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.mask_listing_id(listing_id STRING)
RETURNS STRING
RETURN CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('admins') OR IS_ACCOUNT_GROUP_MEMBER('admin_group') THEN listing_id
    ELSE CONCAT('ID-***', RIGHT(CAST(listing_id AS STRING), 4))
END
""")

# MASK: Price rounding for analysts
spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.mask_exact_price(price_rub DOUBLE)
RETURNS DOUBLE
RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('admins') OR IS_ACCOUNT_GROUP_MEMBER('admin_group') THEN price_rub
    WHEN IS_ACCOUNT_GROUP_MEMBER('analyst_group') THEN ROUND(price_rub / 10000.0, 0) * 10000
    ELSE NULL
END
""")

# Apply masks to production fact table
spark.sql(f"ALTER TABLE {GOLD_SCHEMA}.fact_listings_liquid ALTER COLUMN listing_id SET MASK {SECURITY_SCHEMA}.mask_listing_id")
spark.sql(f"ALTER TABLE {GOLD_SCHEMA}.fact_listings_liquid ALTER COLUMN price_rub SET MASK {SECURITY_SCHEMA}.mask_exact_price")

# ======================================================================================
# 3. RESOURCE USAGE ANALYSIS - FIXED NameError
# ======================================================================================
print("\n📊 Performing Resource Usage Analysis...")
# Variable defined as usage_analysis_df to match display call
usage_analysis_df = spark.sql(f"""
    SELECT query, liquid_avg_secs, zorder_avg_secs,
           ROUND(((zorder_avg_secs - liquid_avg_secs) / zorder_avg_secs) * 100, 2) as efficiency_gain_pct
    FROM {GOLD_SCHEMA}.benchmark_results
""")

display(usage_analysis_df) # Fixed NameError

# ======================================================================================
# 4. FINAL VERIFICATION
# ======================================================================================
print("\n🧪 Live Data Sample (Admins see original, Analysts see masked):")
display(spark.sql(f"SELECT listing_id, price_rub, brand, model, location_key FROM {GOLD_SCHEMA}.fact_listings_liquid LIMIT 10"))

print("\n🎯 Day 8-9: Access Controls and Analysis Complete!")

In [0]:
# src/notebooks/09_unity_catalog_security.py (Continued)
# Branch: feature/gold-layer

# ======================================================================================
# 1. MASKING listing_id: Protecting Reference Identity
# ======================================================================================
# Business logic: Non-admins see only the last 4 characters to prevent mass scraping.
print("🚀 Creating Masking Function for listing_id...")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.mask_listing_id(listing_id STRING)
RETURNS STRING
RETURN
  CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin_group') THEN listing_id
    ELSE CONCAT('ID-***', RIGHT(CAST(listing_id AS STRING), 4))
  END
""")

# Applying MASK to fact_listings_liquid
spark.sql(f"""
ALTER TABLE {GOLD_SCHEMA}.fact_listings_liquid
ALTER COLUMN listing_id SET MASK {SECURITY_SCHEMA}.mask_listing_id
""")
print("✓ listing_id masking applied (non-admins see ID-***XXXX format)")

# ======================================================================================
# 2. MASKING price_rub: Competitive Intelligence Protection
# ======================================================================================
# Logic: Analysts see rounded price to nearest 10k, others see NULL.
# This prevents competitor price intelligence at an exact level.
print("🚀 Creating Masking Function for price_rub...")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.mask_exact_price(price_rub DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin_group') THEN price_rub
    WHEN IS_ACCOUNT_GROUP_MEMBER('analyst_group') THEN ROUND(price_rub / 10000.0, 0) * 10000
    ELSE NULL
  END
""")

# Applying MASK to price_rub
spark.sql(f"""
ALTER TABLE {GOLD_SCHEMA}.fact_listings_liquid
ALTER COLUMN price_rub SET MASK {SECURITY_SCHEMA}.mask_exact_price
""")
print("✓ price_rub masking applied (analysts see rounded price; others see NULL)")

# ======================================================================================
# 3. MASKING metadata/description (Project-specific column)
# ======================================================================================
# Applying to metadata/description column if exists in gold table to hide raw details.
print("🚀 Creating Masking Function for internal_notes/metadata...")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {SECURITY_SCHEMA}.mask_sensitive_notes(notes STRING)
RETURNS STRING
RETURN
  CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('admin_group') THEN notes
    ELSE '[REDACTED - SENSITIVE CONTENT]'
  END
""")

# Applying to existing description columns to maintain security across dimensions
# If using dim_listing_details from our silver layer logic
spark.sql(f"""
ALTER TABLE {GOLD_SCHEMA}.fact_listings_liquid
ALTER COLUMN model SET MASK {SECURITY_SCHEMA}.mask_sensitive_notes
""") # Example: Masking a field to demonstrate functionality
print("✓ Sensitive content masking applied to fact_listings_liquid")

print("\n✅ Column-Level Security (CLS) Implementation Complete!")

In [0]:
# =========================================================
# 1. GRANT PERMISSIONS via Unity Catalog
# =========================================================
print("🚀 Granting SELECT on Gold tables to analyst_group...")

# Project-specific Gold tables from Day 7
gold_tables = [
    "fact_listings_liquid", 
    "agg_stale_inventory", 
    "benchmark_results"
]

for table in gold_tables:
    # analyst_group ab exist karta hai
    spark.sql(f"GRANT SELECT ON TABLE {CATALOG}.gold.{table} TO `analyst_group` ")
    print(f"✓ SELECT granted on gold.{table} to analyst_group")

# Layer Isolation for Security
spark.sql(f"REVOKE ALL PRIVILEGES ON SCHEMA {CATALOG}.bronze FROM `analyst_group` ")
print("✓ Bronze layer access revoked — Security hardened!")

# =========================================================
# 2. VERIFICATION: Audit Security Metadata (Error Fixed)
# =========================================================
print("\n=== 🔍 VERIFYING SECURITY SETUP ===")

# Fix for [UNRESOLVED_COLUMN]: Using standard information_schema columns
print("📊 Applied Security Policies (Table Level):")
display(spark.sql(f"""
    SELECT table_name, table_type, table_schema
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema = 'gold'
"""))

# Verification for Column Masks
print("📊 Applied Column Masks Verification:")
display(spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = 'gold'
    AND table_name = 'fact_listings_liquid'
"""))

# =========================================================
# 3. DEMO: Security Validation (The "Real" Test)
# =========================================================
print("\n=== 🧪 ANALYST VIEW (Security Demo) ===")
# Ye cell prove karega ki masking aur filters active hain

demo_df = spark.sql(f"""
    SELECT 
        listing_id, 
        brand, 
        model, 
        location_key, 
        price_rub, 
        gold_load_dt
    FROM {CATALOG}.gold.fact_listings_liquid
    LIMIT 10
""")

display(demo_df)

print("\n🎯 Day 8-9 Security Implementation Complete & Verified!")